# Global Shipping Chokepoints — 03: The Real Cross-Check

Notebook 02 ranked chokepoints by raw AIS traffic density and found the Danish region
(Oresund Strait) dominating with ~69% of the measured total — 900x the Panama Canal.
That's real, but it measures *how often a ship passed*, not *how much cargo it carried*.

This notebook pulls the IMF's own **PortWatch** dataset — a public, unauthenticated,
real dataset (no API key needed) tracking daily vessel transits and **cargo-carrying
capacity** at the same 28 named global chokepoints, built from real AIS signals on
~90,000 ships. It's the honest independent cross-check this project needs: our own
ship-position density vs. the IMF's own measured cargo capacity, for the same places.

Source: https://portwatch.imf.org/pages/data-and-methodology — public ArcGIS
FeatureServer, verified working with a plain unauthenticated request.

## Map our chokepoints to PortWatch's official IDs

PortWatch names 28 canonical global chokepoints. Eight of our nine from notebook 02
map directly onto theirs (Panama Canal, Bosporus/Bosphorus is the same strait despite
the spelling difference). Our broader "Danish Straits" box maps to their more specific
"Oresund Strait" — one real strait inside that same region, not the identical footprint,
which is worth being precise about rather than pretending they're the same box.

In [0]:
CHOKEPOINT_ID_MAP = {
    "Strait of Malacca":    ("chokepoint5",  "Malacca Strait"),
    "Suez Canal":           ("chokepoint1",  "Suez Canal"),
    "Panama Canal":         ("chokepoint2",  "Panama Canal"),
    "Strait of Hormuz":     ("chokepoint6",  "Strait of Hormuz"),
    "Bab-el-Mandeb":        ("chokepoint4",  "Bab el-Mandeb Strait"),
    "Danish Straits":       ("chokepoint10", "Oresund Strait"),
    "Strait of Dover":      ("chokepoint9",  "Dover Strait"),
    "Strait of Gibraltar":  ("chokepoint8",  "Gibraltar Strait"),
    "Bosphorus Strait":     ("chokepoint3",  "Bosporus Strait"),
}

## Pull the last full year of daily data for each chokepoint

The public FeatureServer needs no auth, just an HTTP GET. We average over a full year
(rather than a single day) to get a stable, representative number instead of one day's
noise — shipping traffic varies a lot day to day.

In [0]:
import requests
import pandas as pd
from datetime import date, timedelta

BASE_URL = "https://services9.arcgis.com/weJ1QsnbMYJlCHdG/ArcGIS/rest/services/Daily_Chokepoints_Data/FeatureServer/0/query"

end_date = date.today() - timedelta(days=14)  # avoid the last ~2 weeks: data can still be backfilling
start_date = end_date - timedelta(days=365)

rows = []
for our_name, (portid, portwatch_name) in CHOKEPOINT_ID_MAP.items():
    params = {
        "where": f"portid='{portid}' AND date >= DATE '{start_date}' AND date <= DATE '{end_date}'",
        "outFields": "date,n_total,capacity",
        "f": "json",
        "resultRecordCount": 1000,
    }
    resp = requests.get(BASE_URL, params=params, timeout=30)
    resp.raise_for_status()
    features = resp.json().get("features", [])
    if not features:
        print(f"WARNING: no data returned for {our_name} ({portid})")
        continue
    daily = pd.DataFrame([f["attributes"] for f in features])
    rows.append({
        "chokepoint": our_name,
        "portwatch_name": portwatch_name,
        "portwatch_days_available": len(daily),
        "avg_daily_vessels": daily["n_total"].mean(),
        "avg_daily_capacity": daily["capacity"].mean(),
    })

portwatch_df = pd.DataFrame(rows)
display(portwatch_df)

chokepoint,portwatch_name,portwatch_days_available,avg_daily_vessels,avg_daily_capacity
Strait of Malacca,Malacca Strait,366,230.21857923497268,8660301.521857923
Suez Canal,Suez Canal,366,41.02459016393443,1493223.0218579236
Panama Canal,Panama Canal,366,31.9672131147541,928148.7486338798
Strait of Hormuz,Strait of Hormuz,366,40.0792349726776,1494023.2486338797
Bab-el-Mandeb,Bab el-Mandeb Strait,366,34.76229508196721,1283542.8715846995
Danish Straits,Oresund Strait,366,43.568306010928964,102347.89617486339
Strait of Dover,Dover Strait,366,168.78415300546447,3472712.150273224
Strait of Gibraltar,Gibraltar Strait,366,133.69945355191257,3521643.6174863386
Bosphorus Strait,Bosporus Strait,366,85.66666666666667,1097813.650273224


## Join with our own traffic-density ranking from notebook 02

This is the actual point of the whole project: put our own measured ship-position
density share next to the IMF's own measured cargo-capacity share, for the same real
places, and let the gap between them speak for itself.

In [0]:
ours = spark.table("workspace.global_shipping.known_chokepoints_ranked").toPandas()

final = ours.merge(portwatch_df, on="chokepoint", how="left")
final["capacity_share"] = final["avg_daily_capacity"] / final["avg_daily_capacity"].sum()
final = final.sort_values("avg_daily_capacity", ascending=False).reset_index(drop=True)

display(final[[
    "chokepoint", "nearest_port", "nearest_port_country",
    "share_of_ranked_total", "avg_daily_vessels", "avg_daily_capacity", "capacity_share",
]].rename(columns={"share_of_ranked_total": "our_traffic_density_share"}))

chokepoint,nearest_port,nearest_port_country,our_traffic_density_share,avg_daily_vessels,avg_daily_capacity,capacity_share
Strait of Malacca,TELUK ANSON,MY,0.13952402891848564,230.21857923497268,8660301.521857923,0.3926905347306774
Strait of Gibraltar,TANGIER-MEDITERRANEAN,MA,0.016908101812973853,133.69945355191257,3521643.6174863386,0.1596845227375993
Strait of Dover,CALAIS,FR,0.05110482230764081,168.78415300546447,3472712.150273224,0.15746578659122176
Strait of Hormuz,KHAWR KHASAB,OM,0.06446823230815885,40.0792349726776,1494023.2486338797,0.06774461454088468
Suez Canal,EL ISMAILIYA,EG,0.00823871965418762,41.02459016393443,1493223.0218579236,0.0677083292591583
Bab-el-Mandeb,ASSAB,ER,0.01646168048119253,34.76229508196721,1283542.8715846995,0.05820064524545705
Bosphorus Strait,ISTINYE,TR,0.007719676126366622,85.66666666666667,1097813.650273224,0.0497789861325687
Panama Canal,VACAMONTE,PA,8.026211227774331E-4,31.9672131147541,928148.7486338798,0.042085743491810344
Danish Straits,HUNDESTED,DK,0.6947721172682166,43.568306010928964,102347.89617486339,0.004640837270622494


## Save the final comparison

In [0]:
result = spark.createDataFrame(final)
result.write.format("delta").mode("overwrite").saveAsTable("workspace.global_shipping.chokepoints_final_comparison")
print("Saved workspace.global_shipping.chokepoints_final_comparison")

Saved workspace.global_shipping.chokepoints_final_comparison
